# Phase 3D — Manual Colab Runner

Runs the **30 frozen Phase 3D experiments** manually in Colab:

- A1, D1, E11, E13, E15
- MOFA+ and SCOT
- seeds 1729, 2718, 31415
- E18 intentionally excluded because ATAC is not verified

This notebook uses the existing repository benchmark code. It does **not** change scientific parameters.
It only reconciles the stale `start_commit` field inside the Colab working copy so the prepared runner can execute from commit `c7732e2`.

Results are saved to Google Drive under `MyDrive/phase3d_manual_results`.


In [ ]:
# 1) Runtime setup
%pip install -q "mofapy2==0.7.5" "POT==0.9.6.post1" gdown

import os, re, json, hashlib, shutil, subprocess, sys, platform, time
from pathlib import Path
from collections import Counter

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/rifatahsanpul0k/research.git"
REPO = Path("/content/multiomics-research")
SOURCE_COMMIT = "c7732e2b916086f17122610f5ce2be991d62e259"
SCOT_COMMIT = "14649be6e14017dcfe7ba619091b33d1df55f6a9"
RESULT_BACKUP = Path("/content/drive/MyDrive/phase3d_manual_results")
RESULT_BACKUP.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Result backup:", RESULT_BACKUP)


In [ ]:
# 2) Clone the exact prepared Phase 3D repository commit
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert HEAD == SOURCE_COMMIT, (HEAD, SOURCE_COMMIT)
print("Repository commit:", HEAD)


In [ ]:
# 3) Download the verified Phase 3D datasets from the project's manifest
import gdown

manifest_path = REPO / "02_omics/01_measurement_and_data_generation/DOWNLOAD_MANIFEST.json"
manifest = json.loads(manifest_path.read_text())

ALLOWED_DATASETS = {
    "10x_human_lymph_node_A1",
    "10x_human_lymph_node_D1",
    "Mouse_Brain_E11_S1",
    "Mouse_Brain_E13_S1",
    "Mouse_Brain_E15_S1",
}

def sha256(path: Path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

complete = [
    r for r in manifest
    if r.get("status") == "complete" and r["dataset_id"] in ALLOWED_DATASETS
]

for rec in complete:
    out = REPO / "04_datasets" / rec["dataset_id"] / "raw" / rec["filename"]
    out.parent.mkdir(parents=True, exist_ok=True)

    if out.exists() and sha256(out) == rec["sha256"]:
        print("verified existing:", rec["dataset_id"], rec["filename"])
        continue

    m = re.search(r"/file/d/([^/]+)", rec["source_file_url"])
    if not m:
        raise RuntimeError(f"Could not extract Drive file ID: {rec['source_file_url']}")
    file_id = m.group(1)

    print("downloading:", rec["dataset_id"], rec["filename"])
    gdown.download(id=file_id, output=str(out), quiet=False)

    observed = sha256(out)
    if observed != rec["sha256"]:
        out.unlink(missing_ok=True)
        raise RuntimeError(
            f"Checksum mismatch for {rec['dataset_id']}/{rec['filename']}\n"
            f"expected={rec['sha256']}\nobserved={observed}"
        )

print(f"Verified {len(complete)} source files.")


In [ ]:
# 4) Install the exact official SCOT source expected by the project wrapper
external_scot = REPO / "external" / "SCOT"
external_scot.parent.mkdir(parents=True, exist_ok=True)

if external_scot.exists():
    shutil.rmtree(external_scot)

subprocess.run(
    ["git", "clone", "https://github.com/rsinghlab/SCOT.git", str(external_scot)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(external_scot), "checkout", "--detach", SCOT_COMMIT],
    check=True,
)

actual = subprocess.check_output(
    ["git", "-C", str(external_scot), "rev-parse", "HEAD"], text=True
).strip()
assert actual == SCOT_COMMIT
print("SCOT commit:", actual)


## Why the next cell edits `start_commit`

The scientific configs were frozen with `start_commit=951a61c`, but the actual Phase 3D implementation was committed later at `c7732e2`.
The repository runner refuses this mismatch before any model executes.

For **manual Colab execution only**, the next cell changes only `start_commit` in the Colab copy of each config to the exact checked-out commit. It does not change dataset, method, seed, preprocessing, MOFA settings, SCOT settings, clustering, or metrics.


In [ ]:
# 5) Reconcile ONLY the stale provenance field in the Colab working copy
CONFIG_DIR = REPO / "07_models/02_classical_integration/configs"
configs = sorted(CONFIG_DIR.glob("*.json"))

assert len(configs) == 30, f"Expected 30 configs, found {len(configs)}"

for path in configs:
    cfg = json.loads(path.read_text())
    old = cfg.get("start_commit")
    cfg["start_commit"] = SOURCE_COMMIT
    cfg["manual_execution_mode"] = "MANUAL_COLAB"
    path.write_text(json.dumps(cfg, indent=2) + "\n")
    print(path.name, old[:8] if old else None, "->", SOURCE_COMMIT[:8])

# Prove all scientific fields still match the intended matrix.
seeds = Counter()
methods = Counter()
datasets = Counter()
for path in configs:
    cfg = json.loads(path.read_text())
    seeds[cfg["seed"]] += 1
    methods[cfg["method"]] += 1
    datasets[cfg["dataset"]] += 1

assert seeds == Counter({1729:10, 2718:10, 31415:10})
assert methods == Counter({"MOFAPLUS":15, "SCOT":15})
assert datasets == Counter({"LN_A1":6, "LN_D1":6, "MB_E11":6, "MB_E13":6, "MB_E15":6})
print("30-run matrix verified.")


In [ ]:
# 6) Colab execution environment required by run_phase3d.py
os.environ["ASTRA_COLAB_CONFIRMED"] = "1"
os.environ["COLAB_RUNTIME_TYPE"] = os.environ.get("COLAB_RELEASE_TAG", "Google-Colab")
os.environ["ASTRA_COLAB_RAM_CLASS"] = "runtime-provided"

runner = REPO / "code/benchmarks/run_phase3d.py"

def backup_experiment(exp_id):
    src = REPO / "08_experiments" / exp_id
    dst = RESULT_BACKUP / exp_id
    if dst.exists():
        shutil.rmtree(dst)
    if src.exists():
        shutil.copytree(src, dst)

def run_config(path):
    cfg = json.loads(path.read_text())
    exp_id = cfg["experiment_id"]

    # Do not rerun a result already backed up successfully.
    saved_status = RESULT_BACKUP / exp_id / "STATUS"
    if saved_status.exists() and saved_status.read_text().strip() == "SUCCEEDED":
        print("SKIP already saved:", exp_id)
        return {"experiment_id": exp_id, "status": "SUCCEEDED", "source": "existing_backup"}

    existing = REPO / "08_experiments" / exp_id
    if existing.exists():
        shutil.rmtree(existing)

    print("\nRUN:", exp_id)
    p = subprocess.run(
        [sys.executable, str(runner), "--config", str(path)],
        cwd=REPO,
        text=True,
        capture_output=True,
    )

    print(p.stdout[-4000:])
    if p.stderr:
        print(p.stderr[-4000:], file=sys.stderr)

    backup_experiment(exp_id)

    status_path = REPO / "08_experiments" / exp_id / "STATUS"
    status = status_path.read_text().strip() if status_path.exists() else "FAILED"
    return {"experiment_id": exp_id, "status": status, "returncode": p.returncode}


## 7. Gate: run one MOFA+ and one SCOT configuration first

Do not continue to the full 30 if either gate fails. Send the displayed error back to ChatGPT.


In [ ]:
gate_names = [
    "EXP-LN-A1-MOFAPLUS-KMEANS-S1729.json",
    "EXP-LN-A1-SCOT-KMEANS-S1729.json",
]
gate_results = [run_config(CONFIG_DIR / name) for name in gate_names]
print(json.dumps(gate_results, indent=2))

if not all(x["status"] == "SUCCEEDED" for x in gate_results):
    raise RuntimeError("Gate failed. Stop here and send the error output to ChatGPT.")


## 8. Run/resume all 30 experiments

Successful gate experiments are skipped because they are already backed up.
The results are copied to Google Drive after every run, so a Colab disconnect does not lose completed runs.


In [ ]:
results = []
for i, path in enumerate(configs, 1):
    print(f"\n========== {i}/30 ==========")
    result = run_config(path)
    results.append(result)

counts = Counter(x["status"] for x in results)
print("\nFinal status counts:", dict(counts))


In [ ]:
# 9) Collect compact metrics into one CSV in Google Drive
import pandas as pd

rows = []
for path in configs:
    cfg = json.loads(path.read_text())
    exp_id = cfg["experiment_id"]
    exp_dir = RESULT_BACKUP / exp_id
    status_path = exp_dir / "STATUS"
    status = status_path.read_text().strip() if status_path.exists() else "MISSING"

    row = {
        "experiment_id": exp_id,
        "dataset": cfg["dataset"],
        "method": cfg["method"],
        "seed": cfg["seed"],
        "status": status,
    }

    metrics_path = exp_dir / "metrics.json"
    prov_path = exp_dir / "provenance.json"
    validation_path = exp_dir / "validation.json"

    if metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        row["ARI"] = m["ARI"]["result"]
        row["NMI"] = m["NMI"]["result"]
        row["silhouette"] = m["silhouette"]["result"]
        row["cluster_count"] = m["cluster_count"]

    if prov_path.exists():
        p = json.loads(prov_path.read_text())
        row["runtime_seconds"] = p.get("runtime_seconds")

    if validation_path.exists():
        v = json.loads(validation_path.read_text())
        row["scientific_qc_status"] = v.get("scientific_qc_status")

    rows.append(row)

df = pd.DataFrame(rows).sort_values(["dataset", "method", "seed"])
out_csv = RESULT_BACKUP / "PHASE_3D_MANUAL_RESULTS.csv"
df.to_csv(out_csv, index=False)

ok = df[df["status"] == "SUCCEEDED"]
agg = ok.groupby(["dataset", "method"], as_index=False).agg(
    runs=("experiment_id", "count"),
    ARI_mean=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    NMI_mean=("NMI", "mean"),
    NMI_std=("NMI", "std"),
    silhouette_mean=("silhouette", "mean"),
    silhouette_std=("silhouette", "std"),
    runtime_mean_seconds=("runtime_seconds", "mean"),
)
agg_csv = RESULT_BACKUP / "PHASE_3D_MANUAL_AGGREGATES.csv"
agg.to_csv(agg_csv, index=False)

display(df)
display(agg)
print("Saved:", out_csv)
print("Saved:", agg_csv)


In [ ]:
# 10) Final source checksum verification + ZIP backup
for rec in complete:
    path = REPO / "04_datasets" / rec["dataset_id"] / "raw" / rec["filename"]
    observed = sha256(path)
    assert observed == rec["sha256"], f"Source changed: {path}"

archive = shutil.make_archive(
    "/content/phase3d_manual_results",
    "zip",
    root_dir=RESULT_BACKUP,
)
print("All dataset source checksums preserved.")
print("ZIP:", archive)
print("Persistent results:", RESULT_BACKUP)
print("\nAfter this finishes, send PHASE_3D_MANUAL_RESULTS.csv and PHASE_3D_MANUAL_AGGREGATES.csv back to ChatGPT.")
